# YOLO를 사용한 카메라 실습

## 실습 절차
0. 환경 설정
    - Source Image 경로, YOLO Dataset을 저장할 경로, 인지할 Class 명 설정
1. YOLO 사전 학습 모델 실행
    - Local Image 또는 Ultralytics에서 제공하는 Sample Image를 사용
2. Dataset 분할
    - Train / Validation Dataset 분할
3. Source Image Data 라벨링
    - 라벨링 툴 또는 YOLO 사전 학습 모델을 사용하여 라벨링
    - 라벨링 결과 검수
4. 모델 학습
    - Dataset 설정 파일 생성
    - 학습 및 추론
5. 부록
    - 데이터가 없을 때 빠른 파이프라인 확인

## 0) 환경 설정

- 실습을 위해 ultralytics, opencv-python, matplotlib, pandas, numpy 패키지가 필요합니다.

- 패키지가 없다면 아래 셀을 실행해 설치하세요.

In [ ]:
# 설치가 필요하면 아래의 %pip 줄 주석을 해제하고 실행하세요.
# %pip install -q ultralytics opencv-python matplotlib pandas numpy
!pip install -q ultralytics opencv-python matplotlib pandas numpy

print("필요한 패키지 설치 완료!")

In [ ]:
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO
from pathlib import Path
import random
import shutil

YOLO_DATASET_ROOT = Path("./camera_dataset/yolo_tiny") # YOLO Dataset을 저장할 경로를 설정
SOURCE_IMAGES_DIR = Path("../data/camera/vehicle") # Source Images가 저장된 경로를 설정
CLASS_NAMES = ["person", "car", "bus", "truck", "bicycle", "motorcycle"] # YOLO 모델로 인지할 Class 명을 설정

print("실습을 위해 필요한 세부 설정 및 패키지 임포트 완료!")
print("[YOLO Dataset을 저장할 경로]:", YOLO_DATASET_ROOT.resolve())

## 1) YOLO 사전 학습 모델 실행
- 입력 Image를 준비한 후, 사전 학습된 YOLO 모델을 사용하여 입력 Image에 대해서 추론을 수행합니다.

### 1-0) 입력 Image 준비
- **Option 1**: 컴퓨터에 저장된 Local Image를 사용합니다.
- **Option 2**: Ultralytics에서 제공하는 Sample Image를 사용합니다.

- +) 아래 셀은 Option 2의 방식을 사용합니다. Option 1의 방식을 사용하려면 Option 1 아래의 주석을 해제하고 셀을 실행하세요.

In [ ]:
# Option 1: Local Image 경로를 직접 지정
# image_file_path = r"C:\\path\\to\\your\\image.jpg"
# image = cv2.imread(image_file_path)

# Option 2: Ultralytics에서 제공하는 Sample Image를 다운로드해서 사용
from urllib.request import urlretrieve
image_remote_path = "https://ultralytics.com/images/bus.jpg"
image_file_path = "bus.jpg"
urlretrieve(image_remote_path, image_file_path)
image = cv2.imread(image_file_path)

print("[입력 Image 경로]:", image_remote_path)
print("[저장한 Image 경로]: ", image_file_path)

# matplotlib으로 입력 Image 시각화
image_rgb = cv2.cvtColor(XXXXX, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(10, 6))
plt.imshow(image_rgb)
plt.axis("off")
plt.title("Input Image")
plt.show()

### 1-1) 사전 학습된 YOLO 모델(YOLO11n)로 추론 수행 

- 가볍고, 추론 속도가 빠른 사전 학습된 YOLO 모델인 `yolo11n.pt`의 가중치를 사용하여 추론을 수행합니다.
 

In [ ]:
# 사전 학습된 YOLO 모델을 불러오고, 입력 Image에 대해서 추론을 수행
model = YOLO("yolo11n.pt")
results = model(XXXXXXXXXXXX, conf=0.3, verbose=True)  # results는 여러 이미지에 대한 예측 결과를 담는 리스트 형태임

# 일반적으로 이미지 하나만 입력할 경우, 리스트의 첫 번째(0번째) 요소가 해당 이미지의 결과임
result = results[0]
# result에 담겨있는 정보의 키(속성) 출력
print("result에 담겨있는 키(속성):", dir(result))

# 결과 출력
inference_ms = result.speed["inference"]
num_boxes = len(result.boxes)
detected_classes = []
for box in result.boxes:
    cls_id = int(box.cls[0].cpu().numpy())
    detected_classes.append(result.names[cls_id])
detected_classes = sorted(set(detected_classes))

# 추론 시간, 검출된 객체 수, 검출된 객체 종류를 텍스트로 출력
print(f"[추론 시간]: {inference_ms:.1f} ms")
print(f"[검출된 객체 수]: {num_boxes}개")

if detected_classes:
    print(f"[검출된 객체 종류]: {', '.join(detected_classes)}")
else:
    print("[검출된 객체 종류]: 없음")

### 1-2) 다른 사전 학습 YOLO 모델로 추론 수행 및 결과 비교

- `1-1` 코드와 동일한 방식으로, 모델 가중치만 바꿔 추론을 수행합니다.
- `ALT_MODEL_NAME` 값을 변경하면 다양한 사전 학습 모델(`yolo11s.pt`, `yolo11m.pt` 등)을 실습할 수 있습니다.
- 아래 비교 셀에서는 `1-1`과 `1-2` 결과를 한 번에 Image와 표로 비교합니다.

In [ ]:
ALT_MODEL_NAME = "yolo11x.pt"  # yolo11x.pt: 가장 크고 정확도가 높은 YOLOv11 모델 (추론 속도는 느림)
# yolo11n.pt: 가장 작은 모델, 매우 빠르나 정확도는 가장 낮음
# yolo11s.pt: 소형 모델, 속도와 정확도의 균형
# yolo11m.pt: 중형 모델, 정확도가 더 높음
# yolo11l.pt: 대형 모델, 더욱 높은 정확도
alt_model = YOLO(ALT_MODEL_NAME)
alt_results = alt_model(XXXXXXXXXXXXX, conf=0.3, verbose=False) # TO DO: alt_model에 입력 Image를 넣어 추론 수행

alt_result = alt_results[0]
alt_inference_ms = alt_result.speed["inference"]
alt_num_boxes = len(alt_result.boxes)
alt_detected_classes = []
for box in alt_result.boxes:
    cls_id = int(box.cls[0].cpu().numpy())
    alt_detected_classes.append(alt_result.names[cls_id])
alt_detected_classes = sorted(set(alt_detected_classes))

print(f"[모델]: {ALT_MODEL_NAME}")
print(f"[추론 시간]: {alt_inference_ms:.1f} ms")
print(f"[검출된 객체 수]: {alt_num_boxes}개")
if alt_detected_classes:
    print(f"[검출된 객체 종류]: {', '.join(alt_detected_classes)}")
else:
    print("[검출된 객체 종류]: 없음")

In [ ]:
# 1-1(YOLO11n) vs 1-2(ALT_MODEL_NAME) 결과를 Image로 비교
base_annotated = result.plot()  # BGR image
base_annotated_rgb = cv2.cvtColor(XXXXXXXXXXX, cv2.COLOR_BGR2RGB)

alt_annotated = alt_result.plot()  # BGR image
alt_annotated_rgb = cv2.cvtColor(XXXXXXXXXXX, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(16, 6))
plt.subplot(1, 2, 1)
plt.imshow(base_annotated_rgb)
plt.axis("off")
plt.title("yolo11n.pt Result")

plt.subplot(1, 2, 2)
plt.imshow(alt_annotated_rgb)
plt.axis("off")
plt.title(f"{ALT_MODEL_NAME} Result")

plt.tight_layout()
plt.show()

In [ ]:
# 1-1(YOLO11n) vs 1-2(ALT_MODEL_NAME) 결과를 표(DataFrame)로 비교
def boxes_to_df(det_result):
    rows = []
    for box in det_result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        conf = float(box.conf[0].cpu().numpy())
        cls_id = int(box.cls[0].cpu().numpy())
        cls_name = det_result.names[cls_id]
        rows.append({
            "class": cls_name,
            "conf": conf,
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "w": x2 - x1,
            "h": y2 - y1,
            "cx": (x1 + x2) / 2,
            "cy": (y1 + y2) / 2,
        })
    if len(rows) == 0:
        return pd.DataFrame(columns=["class", "conf", "x1", "y1", "x2", "y2", "w", "h", "cx", "cy"])
    return pd.DataFrame(rows).sort_values("conf", ascending=False).reset_index(drop=True)

df_base = boxes_to_df(result)
df_alt = boxes_to_df(alt_result)

summary_df = pd.DataFrame([
    {
        "model": "yolo11n.pt",
        "inference_ms": inference_ms,
        "num_boxes": num_boxes,
        "detected_classes": ", ".join(detected_classes) if detected_classes else "없음",
    },
    {
        "model": ALT_MODEL_NAME,
        "inference_ms": alt_inference_ms,
        "num_boxes": alt_num_boxes,
        "detected_classes": ", ".join(alt_detected_classes) if alt_detected_classes else "없음",
    },
])

print("[모델별 요약 비교]")
display(summary_df)

print("\n[yolo11n.pt 상위 검출 결과]")
display(df_base.head(10))

print(f"\n[{ALT_MODEL_NAME} 상위 검출 결과]")
display(df_alt.head(10))

## 2) Dataset 분할

- `SOURCE_IMAGES_DIR`에 저장되어 있는 Image를 YOLO_DATASET_ROOT로 복사하고 train / val / test로 분할합니다.
    - 파일명에 `test`가 포함된 이미지는 자동으로 `images/test`에 복사됩니다.
- 라벨링 시작 전에 이 셀을 먼저 **1회** 실행하세요.

In [ ]:
# ======================
# 1단계: 파라미터 설정
# ======================
TRAIN_RATIO = 0.8               # train 이미지 비율
RANDOM_SEED = 42                # 무작위 분할 seed (재현성 확보용)
CLEAR_OLD_SPLIT_IMAGES = True   # True이면 기존 train/val 이미지파일 삭제 후 다시 분할
CLEAR_OLD_TEST_IMAGES = True    # True이면 기존 test 이미지파일 삭제 후 다시 구성

# ==============================
# 2단계: 전체 이미지 목록 수집
# ==============================
exts = {".jpg", ".jpeg", ".png", ".bmp"}    # 사용할 이미지 확장자
all_images = [path for path in SOURCE_IMAGES_DIR.rglob("*") 
              if path.suffix.lower() in exts] if SOURCE_IMAGES_DIR.exists() else []

# ======================
# 3단계: 폴더 준비
# ======================
train_img_dir = YOLO_DATASET_ROOT / "images" / "train"
train_lbl_dir = YOLO_DATASET_ROOT / "labels" / "train"
val_img_dir   = YOLO_DATASET_ROOT / "images" / "val"
val_lbl_dir   = YOLO_DATASET_ROOT / "labels" / "val"
test_img_dir  = YOLO_DATASET_ROOT / "images" / "test"

for path in [train_img_dir, train_lbl_dir, val_img_dir, val_lbl_dir, test_img_dir]:
    path.mkdir(parents=True, exist_ok=True)    # 필요한 디렉토리를 모두 생성

# =====================================================
# 4단계: 파일명에 "test"가 포함된 이미지를 test set으로 분리
# =====================================================
test_images = [path for path in all_images if "test" in path.name.lower()]
split_candidates = [path for path in all_images if "test" not in path.name.lower()] # 나머지는 train/val용 후보

# ========================================
# 5단계: 기존 이미지 파일 모두 삭제 (옵션)
# ========================================
if CLEAR_OLD_SPLIT_IMAGES:
    for old in list(train_img_dir.glob("*")) + list(val_img_dir.glob("*")):
        if old.is_file():
            old.unlink()    # 이전 train/val 이미지 삭제

if CLEAR_OLD_TEST_IMAGES:
    for old in list(test_img_dir.glob("*")):
        if old.is_file():
            old.unlink()    # 이전 test 이미지 삭제

# ========================================
# 6단계: test set 이미지 분리/복사
# ========================================
if len(test_images) > 0:
    for src in test_images:
        shutil.copy2(src, test_img_dir / src.name)

# ========================================
# 7단계: 남은 이미지 train/val로 분할
# ========================================
if len(split_candidates) == 0:
    print("[WARNING] train/val로 분할할 원본 이미지가 없음!:", SOURCE_IMAGES_DIR)
else:
    random.seed(RANDOM_SEED)
    random.shuffle(split_candidates)  # 셔플

    n_train = max(1, int(len(split_candidates) * TRAIN_RATIO))
    # 최소 1개~(전체-1)개로 제한 (이미지가 2장일 때 val도 최소 1장 되도록)
    n_train = min(n_train, len(split_candidates) - 1) if len(split_candidates) > 1 else 1 

    train_imgs = split_candidates[:n_train]
    val_imgs = split_candidates[n_train:]

    # 각각 복사
    for src in train_imgs:
        shutil.copy2(src, train_img_dir / src.name)
    for src in val_imgs:
        shutil.copy2(src, val_img_dir / src.name)

    print(f"[복사 및 분할 완료]: Train={len(train_imgs)} | Val={len(val_imgs)} | Test={len(test_images)}")

print("\n[경로 확인]")
print("SOURCE_IMAGES_DIR:", SOURCE_IMAGES_DIR)
print("YOLO_DATASET_ROOT:", YOLO_DATASET_ROOT)
print("\n[현재 Dataset 상태]")
print("Train images:", len(list(train_img_dir.glob("*"))))
print("Train labels:", len(list(train_lbl_dir.glob("*.txt"))))
print("Val images:", len(list(val_img_dir.glob("*"))))
print("Val labels:", len(list(val_lbl_dir.glob("*.txt"))))
print("Test images:", len(list(test_img_dir.glob("*"))))

## 3) Source Image Data 라벨링
- 먼저 라벨링을 수행할 Image를 시각화하여 살펴봅니다.
- 이후, `LabelImg`라는 라벨링 툴 또는 사전 학습된 YOLO 모델을 사용하여 라벨링을 수행합니다.

### 3-0) 라벨링을 수행 할 Image 살펴보기

In [ ]:
# ----------- [Train 이미지 미리보기] -----------
# - 학습(train) 이미지가 어떻게 생겼는지 빠르게 시각화하여 확인합니다.
# - (분할이 안 됐다면 먼저 데이터 분할 셀 실행 권장)
preview_dir = YOLO_DATASET_ROOT / "images" / "train"  # train 이미지 폴더 경로
valid_exts = {".jpg", ".jpeg", ".png", ".bmp"}  # 미리볼 이미지 확장자 리스트
# train 폴더 내 이미지 파일 6개까지만 미리보기 대상으로 가져옴
preview_imgs = sorted([path for path in preview_dir.glob("*") if path.suffix.lower() in valid_exts])[:6] if preview_dir.exists() else []

if len(preview_imgs) == 0:
    print("[WARNING] 살펴 볼 Train Image가 없습니다. 먼저 Dataset 분할 셀을 실행하세요!")
else:
    plt.figure(figsize=(14, 8))  # figure 크기 (폭 × 높이)
    for i, path in enumerate(preview_imgs, start=1):  # 1번부터 index 시작
        img = cv2.imread(str(path))  # 이미지 읽어오기
        if img is None:
            continue  # 파일 손상 등 예외 케이스 skip
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # OpenCV는 BGR임 → Matplotlib용 RGB 변환
        plt.subplot(2, 3, i)  # 2행 3열 subplot에 그림 표시
        plt.imshow(img)
        plt.title(path.name)  # 파일명 타이틀로 표시
        plt.axis("off")  # 축 눈금 제거
    plt.tight_layout()  # subplot 간격 자동조정
    plt.show()  # 전체 그림 보여주기

### 3-1) 라벨링 수행
- `images/train` 및 `images/val`에 있는 각 이미지에 대해 YOLO 형식의 txt 라벨 파일을 생성합니다.
- Option 1: `LabelImg`라는 라벨링 툴을 사용하여 라벨링을 수행합니다.
- Option 2: 사전 학습된 YOLO 모델을 사용하여 라벨링을 수행합니다.

#### 3-1-1) **Option 1**: `LabelImg`라는 라벨링 툴을 사용하여 라벨링을 수행
1. LabelImg 툴을 설치 및 실행합니다.
2. 좌측의 **Open Dir**를 클릭하여 `practice/camera_dataset/yolo_tiny/images/train` 폴더를 엽니다.
3. 좌측의 **Change Save Dir**를 클릭하여 라벨링 결과 저장 경로를 `practice/camera_dataset/yolo_tiny/labels/train`로 설정합니다.
4. 우측 하단의 **File List**에 있는 파일 선택하여 Image를 불러옵니다.
5. 좌측의 **Save** 아래에 있는 **CreateML**을 클릭하여 라벨링 형식을 YOLO로 변경합니다. (CreateML -> PascalVOC -> YOLO 순으로 변경)
6. 좌측의 **Create RectBox**를 클릭한 후, 객체 주변에 Bounding Box를 그리고 Class 명을 입력합니다. 이후 좌측의 **Save** 버튼을 눌러 저장합니다.
7. Validation Dataset `val`에 대해서도 동일한 과정을 수행합니다. (경로만 `images` -> `labels`로 변경)


In [ ]:
# LabelImg 설치
!pip install labelImg

# LabelImg 실행
# 1. Ubuntu: 터미널에서 `labelImg` 실행
# 2. Windows: Anaconda Prompt에서 `labelImg` 실행
!labelImg

#### 3-1-2) **Option 2**: 사전 학습된 YOLO 모델을 사용하여 라벨링을 수행
- 직접 Bounding Box를 처음부터 수작업으로 그리는 대신, 사전 학습된 YOLO 모델의 예측 결과를 라벨 초안으로 생성한 뒤 사람이 검수 및 수정하는 방식입니다.
- **장점**: 라벨링 시간 단축
- **단점**: 오검출 및 누락이 발생할 수 있으므로 반드시 검수 필요

In [ ]:
# 1. "사전 학습된 YOLO 모델 로드" (작은 모델 사용)
pseudo_model = YOLO("yolo11n.pt")

# 2. 클래스명(문자)을 내부 class id(숫자)로 변환하는 매핑 생성
name_to_id = {name: i for i, name in enumerate(CLASS_NAMES)}  # 내 데이터셋 기준 class 매핑!

# 3. YOLO 모델 예측 결과(boxes)를 실제 txt 라벨 포맷으로 저장하는 함수
def write_yolo_label_file(txt_path: Path, det_boxes, img_w, img_h, names):
    """
    txt_path: 저장할 라벨 txt 파일 경로
    det_boxes: 감지된 bounding box들의 모음 (YOLO 결과)
    img_w, img_h: 원본 이미지의 가로/세로(px)
    names: YOLO가 사용하는 클래스 리스트(문자)
    """
    lines = []
    for box in det_boxes:
        cls_id_coco = int(box.cls[0].cpu().numpy())               # COCO class id (YOLO가 기준)
        cls_name = names[cls_id_coco]                             # class 이름
        if cls_name not in name_to_id:
            continue  # 내 class에 없는 것은 건너뜀

        # box 좌표: x1, y1, x2, y2 (좌상단/우하단 절대좌표)
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()
        
        # YOLO label 포맷(cx, cy, w, h) (모두 0~1로 정규화)
        x_center = ((x1 + x2) / 2.0) / img_w
        y_center = ((y1 + y2) / 2.0) / img_h
        w = (x2 - x1) / img_w
        h = (y2 - y1) / img_h

        local_cls = name_to_id[cls_name]  # 내 데이터셋 class id
        lines.append(f"{local_cls} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}")

    # txt 파일 저장 (한 줄에 bbox 하나)
    txt_path.write_text("\n".join(lines), encoding="utf-8")

# 4. train/val 폴더별로 반복 (실제 자동 라벨 생성 부분)
for split in ["train", "val"]:
    img_dir = YOLO_DATASET_ROOT / "images" / split       # 이미지 폴더
    lbl_dir = YOLO_DATASET_ROOT / "labels" / split       # 라벨 폴더
    img_paths = sorted([path for path in img_dir.iterdir() if path.suffix.lower() in exts])

    for img_path in img_paths:
        # (1) 이미지마다 YOLO 추론
        results = pseudo_model(str(XXXXXXXXX), conf=0.35, verbose=False)
        result = results[0]
        image_height, image_width = result.orig_shape    # 원본 크기 (h, w)
        label_file_path = lbl_dir / f"{img_path.stem}.txt" # txt 저장경로

        # (2) 라벨 txt 파일 작성
        write_yolo_label_file(label_file_path, result.boxes, image_width, image_height, result.names)

print("사전 학습된 YOLO 모델을 사용한 자동 라벨 생성 완료!")
print("※ 주의: 반드시 라벨 파일을 직접 확인/수정(검수)하세요!")

#### 3-1-3) 라벨링 결과 파일 형태 확인

- 라벨링이 끝나면 `images/{split}`와 `labels/{split}`에 같은 파일명 기준으로 이미지-라벨 파일이 짝을 이룹니다.
- 라벨 파일(`.txt`)의 각 줄은 아래 형식 따릅니다.
  - `class_id x_center y_center width height`
  - 좌표 값은 `0~1` 범위의 정규화 값입니다.
- 아래 셀을 실행하면 생성된 라벨 파일 목록과 샘플 파일 내용을 확인할 수 있습니다.

In [ ]:
# 데이터셋 분할 중 미리보기로 사용할 split을 지정합니다("train" 또는 "val").
PREVIEW_SPLIT = "train"

# 미리보기 이미지와 라벨 폴더 경로를 지정합니다.
preview_img_dir = YOLO_DATASET_ROOT / "images" / PREVIEW_SPLIT
preview_lbl_dir = YOLO_DATASET_ROOT / "labels" / PREVIEW_SPLIT

# 허용하는 이미지 확장자 리스트 정의
valid_exts = {".jpg", ".jpeg", ".png", ".bmp"}

# 이미지 파일 목록 불러오기 (폴더가 없으면 빈 리스트 반환)
img_files = sorted([path for path in preview_img_dir.glob("*") if path.suffix.lower() in valid_exts]) if preview_img_dir.exists() else []

# 라벨(txt) 파일 목록 불러오기 (폴더가 없으면 빈 리스트 반환)
lbl_files = sorted(preview_lbl_dir.glob("*.txt")) if preview_lbl_dir.exists() else []

# 현재 split, 이미지/라벨 폴더 경로 및 파일 개수 출력
print(f"[Split] {PREVIEW_SPLIT}")
print("Image 폴더:", preview_img_dir)
print("Label 폴더:", preview_lbl_dir)
print(f"Image 파일 수: {len(img_files)}")
print(f"Label 파일 수: {len(lbl_files)}")

# 라벨 파일이 하나도 없으면 경고 메시지 출력
if len(lbl_files) == 0:
    print("\n[WARNING] 라벨 파일이 없습니다. 라벨링 셀을 먼저 실행하세요.")
else:
    # 라벨 파일 예시로 첫 파일을 출력 (내용 5줄 미리보기)
    sample_label = lbl_files[0]
    print(f"\n[라벨 파일 샘플] {sample_label.name}")
    lines = sample_label.read_text(encoding="utf-8").splitlines()
    if len(lines) == 0:
        print("(빈 파일)")
    else:
        for i, line in enumerate(lines[:5], start=1):
            print(f"{i:02d}: {line}")
        if len(lines) > 5:
            print(f"... (총 {len(lines)}줄)")

# YOLO 라벨 포맷 예시/설명 출력
print("\n[YOLO 라벨 포맷 설명]")
print("각 줄 형식: class_id x_center y_center width height")
print("예) 1 0.512300 0.477100 0.211000 0.325000")

# 이미지 파일과 라벨 파일이 모두 있을 때 (짝이 맞는지) 매칭 예시 출력 (최대 5개만)
if len(img_files) > 0 and len(lbl_files) > 0:
    print("\n[이미지-라벨 매칭 예시]")
    lbl_stems = {path.stem for path in lbl_files}
    shown = 0
    for img_path in img_files:
        has_label = img_path.stem in lbl_stems
        print(f"- {img_path.name} -> {img_path.stem + '.txt'} ({'OK' if has_label else 'MISSING'})")
        shown += 1
        if shown >= 5:
            break
     

### 3-2) 라벨링 결과 검수

- 학습 전에 라벨링 결과를 시각화하여 검수합니다.
- 아래 사항을 중점적으로 점검합니다.
    - Bounding Box가 객체를 정확히 포함하는지
    - Bounding Box가 누락되거나 과다하게 그려지지 않았는지
    - Class 라벨이 올바르게 지정되었는지
- 아래 셀은 Train Dataset `train`에서 랜덤하게 샘플을 뽑아 라벨링 결과를 시각화합니다.

In [ ]:
import math

# YOLO 라벨 txt 파일을 읽어서 (class_id, x_center, y_center, width, height)의 튜플 리스트로 반환하는 함수
def parse_yolo_label_file(label_path: Path):
    rows = []
    if not label_path.exists():
        return rows
    for line in label_path.read_text(encoding="utf-8").splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            continue
        try:
            cls_id = int(parts[0])
            x, y, w, h = map(float, parts[1:])
            rows.append((cls_id, x, y, w, h))
        except ValueError:
            continue
    return rows

# 검수할 Data의 경로 설정
split = "train"
img_dir = YOLO_DATASET_ROOT / "images" / split
lbl_dir = YOLO_DATASET_ROOT / "labels" / split
valid_exts = {".jpg", ".jpeg", ".png", ".bmp"}
img_paths = [p for p in img_dir.glob("*") if p.suffix.lower() in valid_exts] if img_dir.exists() else []

if len(img_paths) == 0:
    print("검수할 이미지가 없습니다. 먼저 Dataset 분할 및 라벨링을 진행하세요!")
else:
    k = min(6, len(img_paths))
    sample_paths = random.sample(img_paths, k)

    # 라벨링 결과 시각화
    plt.figure(figsize=(16, 5 * math.ceil(k / 3)))
    for i, img_path in enumerate(sample_paths, start=1):
        img = cv2.imread(str(img_path))
        if img is None:
            continue
        h, w = img.shape[:2]

        label_path = lbl_dir / f"{img_path.stem}.txt"
        labels = parse_yolo_label_file(label_path)

        for cls_id, x, y, bw, bh in labels:
            x1 = int((x - bw / 2) * w)
            y1 = int((y - bh / 2) * h)
            x2 = int((x + bw / 2) * w)
            y2 = int((y + bh / 2) * h)
            cls_name = CLASS_NAMES[cls_id] if 0 <= cls_id < len(CLASS_NAMES) else f"cls_{cls_id}"
            cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(img, cls_name, (x1, max(0, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.subplot(math.ceil(k / 3), 3, i)
        plt.imshow(img)
        plt.title(f"{img_path.name} | labels={len(labels)}")
        plt.axis("off")

    plt.tight_layout()
    plt.show()

## 4) 모델 학습

### 4-0) Dataset 설정 파일(`data.yaml`) 생성
- YOLO 모델 학습에는 Dataset 경로와 Class 명을 정의한 `data.yaml`이 필요합니다.

In [ ]:
# 1. YOLO 학습에 필요한 data.yaml을 저장할 경로를 지정합니다.
#    이 파일에는 데이터셋의 루트 경로, 학습/검증 이미지 폴더 경로, 클래스 이름이 정의됩니다.
data_yaml_path = YOLO_DATASET_ROOT / "data.yaml"

# 2. data.yaml의 기본 구조를 문자열로 만듭니다.
#    - 'path' 항목은 전체 이미지/라벨 디렉토리들이 있는 루트 경로입니다.
#    - 'train'과 'val'은 각각 학습/검증 이미지 폴더의 상대 경로입니다.
#    - 'names'는 각 클래스 번호별 이름을 매핑합니다.
data_yaml = f"""path: {YOLO_DATASET_ROOT.resolve().as_posix()}
train: images/train
val: images/val

names:
"""

# 3. CLASS_NAMES 순회하며 클래스 정보를 'names' 아래에 한 줄씩 추가합니다.
#    예) 0: bus, 1: person 이런 식으로
for i, name in enumerate(CLASS_NAMES):
    data_yaml += f"  {i}: {name}\n"

# 4. 완성된 data.yaml 내용을 지정한 경로에 'utf-8' 인코딩으로 저장합니다.
data_yaml_path.write_text(data_yaml, encoding="utf-8")

# 5. 파일 저장이 완료되었음을 출력합니다.
print("Dataset 설정 파일 생성 완료!")

# 6. 설정 파일 경로와 실제 파일 내용을 확인차 콘솔에 출력합니다.
print("[Dataset 설정 파일 경로]", data_yaml_path)
print("[Dataset 설정 파일 내용]")
print(data_yaml)

### 4-1) 모델 학습 수행

- 아래 설정을 기준으로 모델 학습을 수행합니다.
    - epochs: 100
    - imgsz: 640
    - model: `yolo11n.pt`
- +) Data가 매우 적으면 과적합이 쉽게 발생합니다.

In [ ]:
# {Model}.pt 구성을 넣으면 프리트레인으로 시작  | {Model}.yaml 구성을 넣으면 랜덤 초기화(프리트레인 X)로 시작
train_model = YOLO("XXXXXXXXXXXX")

# 아래 각 인자는 다음과 같은 의미를 가집니다:
# - data: 학습/검증 데이터셋의 설정 파일(data.yaml) 경로 (데이터셋 구조 및 클래스 정보 포함)
# - epochs: 학습 시 전체 데이터셋을 몇 번 반복할지(에폭 수)
# - imgsz: 입력 이미지의 크기(픽셀 단위)
# - batch: 한 번에 처리할 데이터 배치 크기
# - project: 학습 결과(모델, 로그 등) 저장할 폴더 경로
# - name: 해당 실험(run)의 하위 폴더명
train_results = train_model.train(
    data=str(XXXXXXXXXXXXX),        # 데이터셋 구성 정보
    epochs=10,                     # 총 학습 반복 횟수
    imgsz=640,                      # 입력 이미지 크기
    batch=8,                        # 배치 크기
    project="runs_camera_yolo",     # 결과 저장 프로젝트 폴더
    name="tiny_custom_train",       # 이번 실험의 이름(하위폴더명)
)

print("YOLO 모델 학습 완료!")

### 4-2) 학습한 모델로 검증 및 추론 수행

- 학습이 완료되면, 생성된 `best.pt` 가중치를 사용하여 검증 및 추론을 수행합니다.
- Validation Dataset `val`으로 기본 성능을 평가합니다.
- 모델 추론을 통해 객체 인지 결과를 시각적으로 확인합니다.

In [ ]:
# 1. 학습된 모델의 best.pt 가중치 파일 찾기
run_dir = Path(getattr(train_results, "save_dir", ""))   # train 결과 저장 디렉토리(ex: runs_camera_yolo/...) 객체 생성
candidates = []
if str(run_dir):
    # 현재 실험 결과 폴더(run_dir) 아래 weights/best.pt 직접 추가 (우선순위)
    candidates.append(run_dir / "weights" / "best.pt")
# 전체 runs 폴더 내 best.pt 파일들을 최신 파일 순으로 다 모으기 (백업용, 실험별 이동 순)
candidates += sorted(Path("runs").rglob("best.pt"), key=lambda path: path.stat().st_mtime, reverse=True)
# 실제로 존재하는 best.pt 파일 중 첫 번째(최신/우선순위 높은 것) 선택
best_pt = next((path for path in candidates if path.exists()), None)

# 2. 검증 이미지/테스트 이미지 폴더 내 파일 탐색 (유효 확장자만)
valid_exts = {".jpg", ".jpeg", ".png", ".bmp"}
val_img_dir = YOLO_DATASET_ROOT / "images" / "val"
val_candidates = sorted([path for path in val_img_dir.glob("*") if path.suffix.lower() in valid_exts]) if val_img_dir.exists() else []

test_img_dir = YOLO_DATASET_ROOT / "images" / "test"
test_candidates = sorted([path for path in test_img_dir.glob("*") if path.suffix.lower() in valid_exts]) if test_img_dir.exists() else []
# 테스트 이미지가 있으면 첫 번째 파일 경로 사용, 없으면 None
test_image_file_path = str(test_candidates[0]) if len(test_candidates) > 0 else None

# 3. best.pt(학습된 가중치)가 있으면, 해당 모델로 검증 및 테스트 이미지 예측/시각화
if best_pt is not None:
    print("[best.pt 파일 경로]:", best_pt)
    trained_model = YOLO(str(best_pt))  # 학습된 best.pt 가중치로 모델 객체 생성

    # (1) 검증 이미지(val) 예측 및 그리기
    if len(val_candidates) == 0:
        print("[WARNING] 검증 Image가 없습니다. images/val 폴더를 확인하세요!")
    else:
        # 여러 검증 이미지에 대해 한 번에 추론 진행 (conf=0.25: 최소 신뢰도 0.25 이상만 표시)
        val_results = trained_model([str(path) for path in val_candidates], conf=0.25, verbose=False)

        # 결과를 2열씩 subplot으로 시각화
        cols = 2
        rows = (len(val_candidates) + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(14, 5 * rows))
        axes = axes.flatten() if hasattr(axes, "flatten") else [axes]  # 1D 배열로 변환

        for i, (path, val_result) in enumerate(zip(val_candidates, val_results)):
            vis = val_result.plot()  # 추론 결과를 BGR 형식 이미지로 반환
            vis = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)  # RGB로 변환(plt용)
            axes[i].imshow(vis)
            axes[i].set_title(f"Trained Model Validation Result: {path.name}")
            axes[i].axis("off")

        # 남는 subplot(이미지 없는 칸)은 숨기기
        for j in range(len(val_candidates), len(axes)):
            axes[j].axis("off")

        plt.tight_layout()
        plt.show()

    # (2) 테스트 이미지(test) 예측 및 시각화
    if test_image_file_path is None:
        print("[WARNING] 테스트 Image가 없습니다. images/test 폴더를 확인하세요!")
    else:
        # 테스트 이미지 1개 예측
        test_results = trained_model(test_image_file_path, conf=0.25)
        test_vis = test_results[0].plot()
        test_vis = cv2.cvtColor(test_vis, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(12, 7))
        plt.imshow(test_vis)
        plt.axis("off")
        plt.title("Trained Model Inference Result")
        plt.show()
else:
    print("best.pt를 찾지 못했습니다.")
    print("- train 셀을 먼저 실행했는지 확인하세요.")
    print("- 학습 로그의 save_dir 경로를 확인하세요.")
    print("- 예: runs/detect/.../weights/best.pt")

## 5. 부록) 데이터가 없을 때 빠르게 파이프라인 확인하기

- 직접 라벨링한 데이터가 아직 준비되지 않았다면, 공개 샘플 데이터셋을 활용하여 학습 파이프라인이 정상적으로 동작하는지 먼저 확인할 수 있습니다.
- 이후 직접 생성한 `data.yaml`로 다시 학습하면 됩니다.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import urllib.request
import zipfile
import yaml

# Prepare a fresh coco8 directory to avoid broken previous extractions.
dataset_root = (Path.cwd() / "datasets" / "coco8_fresh").resolve()
dataset_root.mkdir(parents=True, exist_ok=True)

zip_path = dataset_root / "coco8.zip"
zip_url = "https://ultralytics.com/assets/coco8.zip"
if not zip_path.exists():
    urllib.request.urlretrieve(zip_url, zip_path)

yaml_path = dataset_root / "coco8.yaml"
dataset_images_root = dataset_root / "coco8"
if not yaml_path.exists():
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dataset_root)

if not yaml_path.exists():
    nested_yaml = dataset_root / "coco8" / "coco8.yaml"
    if nested_yaml.exists():
        yaml_path = nested_yaml

if not yaml_path.exists():
    # Some coco8 archives may not include coco8.yaml, so create it.
    model_for_names = YOLO("yolo11n.pt")
    names = {int(k): v for k, v in model_for_names.model.names.items()}
    generated_yaml = {
        "path": dataset_images_root.as_posix(),
        "train": "images/train",
        "val": "images/val",
        "names": names,
    }
    yaml_path.write_text(yaml.safe_dump(generated_yaml, sort_keys=False), encoding="utf-8")

if not (dataset_images_root / "images" / "train").exists():
    raise FileNotFoundError(f"train images missing at {(dataset_images_root / 'images' / 'train')}")

train_model = YOLO("yolo11n.pt")
train_model.train(data=str(yaml_path), epochs=3, imgsz=640)

---
### 실습: 공부한 내용을 바탕으로 직접 YOLO 객체 인지 수행해보기

이번 실습에서는 금일 배운 내용을 바탕으로, 준비된 이미지 폴더(`data/camera/practice`) 안의 여러 이미지에 대해 YOLO 모델을 사용해 객체 인식(inference)을 수행하고 그 결과를 시각화합니다.

1. 먼저, `YOLO` 모델을 불러옵니다.
2. `data/camera/practice` 폴더 안의 이미지를 대상으로 예측을 실행합니다.
3. 예측 결과를 이미지 상에 시각화하여 한눈에 확인해봅니다.


In [ ]:
from ultralytics import YOLO
from pathlib import Path
import cv2
import matplotlib.pyplot as plt

# 1) Folder path only (change this one line if needed)
image_folder = Path("../data/camera/practice").resolve()

# 2) Collect images from the folder
image_paths = sorted([*image_folder.glob("*.png"), *image_folder.glob("*.jpg"), *image_folder.glob("*.jpeg")])
if len(image_paths) == 0:
    raise FileNotFoundError(f"No images found in: {image_folder}")
print(f"[Input folder] {image_folder}")
print(f"[Image count] {len(image_paths)}")

# 3) Run YOLO inference for all images
model = XXXXXXXXXXXXXXX
confidence_threshold = 0.25
results = model.predict(
    XXXXXXXXXXXXXXXXXXXXX,
    XXXXXXXXXXXXXXXXXXXXX,
)

# 4) Print per-image detection counts
print("\n[Per-image detection counts]")
total_objects = 0
for result in results:
    n = 0 if result.boxes is None else len(result.boxes)
    total_objects += n
    print(f"- {Path(result.path).name}: {n} objects")
print(f"[Total detected objects] {total_objects}")

# 5) Show all results in a grid
cols = 3
rows = (len(results) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(15, 5 * rows))
axes = axes.flatten()
for i, result in enumerate(results):
    vis = XXXXXXXXXXXXX  # BGR image
    vis = XXXXXXXXXXXXX  # Convert BGR to RGB
    axes[i].imshow(vis)
    axes[i].set_title(Path(result.path).name)
    axes[i].axis("off")
for j in range(len(results), len(axes)):
    axes[j].axis("off")
plt.tight_layout()
plt.show()
print("\n[Saved prediction images]")
print((Path.cwd() / "runs" / "detect" / "runs" / "detect" / "practice_folder11_demo").resolve())